In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
import os
import cv2
import pandas as pd
import numpy as np
from PIL import Image
from tqdm import tqdm

In [ ]:
NIH_PATH = "/kaggle/input/chest-xray14/images"
IU_IMG_PATH = "/kaggle/input/openi/images"
IU_REPORT_PATH = "/kaggle/input/openi/reports"
COVID_PATH = "/kaggle/input/covidx-cxr2"

In [ ]:
for root, dirs, files in os.walk("/kaggle/input"):
    print(root)
    print(files[:5])
    print("-"*50)

In [ ]:
# NIH metadata creation

In [ ]:
import pandas as pd

nih_df = pd.read_csv(
    "/kaggle/input/datasets/biditdas06/nih-chestxray14/Data_Entry_2017_v2020.csv"
)

nih_df.head()

In [ ]:
nih_df.info()

In [ ]:
nih_df.columns

In [ ]:
nih_metadata = nih_df[["Image Index", "Finding Labels"]]
nih_metadata = nih_metadata.rename(columns={
    "Image Index": "image_name",
    "Finding Labels": "disease_labels"
})

nih_metadata.head()

In [ ]:
import os

nih_root = "/kaggle/input/datasets/biditdas06/nih-chestxray14"

image_paths = {}

for root, dirs, files in os.walk(nih_root):
    for file in files:
        if file.endswith(".png"):
            image_paths[file] = os.path.join(root, file)

nih_metadata["image_path"] = nih_metadata["image_name"].map(image_paths)

nih_metadata.head()

In [ ]:
print("Missing image paths:", nih_metadata["image_path"].isnull().sum())

In [ ]:
nih_metadata.to_csv("nih_metadata.csv", index=False)

print("NIH metadata saved successfully!")

# IUXRay metadata creation

In [ ]:
import os
import xml.etree.ElementTree as ET
import pandas as pd

In [ ]:
REPORT_PATH = "/kaggle/input/datasets/biditdas06/iu-x-ray/nlm_cxr_reports/ecgen-radiology"
IMAGE_PATH = "/kaggle/input/datasets/biditdas06/iu-x-ray/nlm_cxr_png"

In [ ]:
report_files = [f for f in os.listdir(REPORT_PATH) if f.endswith(".xml")]

print("Total XML Reports:", len(report_files))

In [ ]:
sample = report_files[0]

tree = ET.parse(os.path.join(REPORT_PATH, sample))
root = tree.getroot()

print(ET.tostring(root, encoding="unicode")[:2000])

In [ ]:
for elem in root.iter():
    print(elem.tag, elem.attrib)

In [ ]:
records = []

for xml_file in report_files:

    tree = ET.parse(os.path.join(REPORT_PATH, xml_file))
    root = tree.getroot()

    findings = ""
    impression = ""

    # Extract Findings and Impression
    for elem in root.iter("AbstractText"):
        label = elem.attrib.get("Label", "")

        if label == "FINDINGS":
            findings = elem.text if elem.text else ""

        elif label == "IMPRESSION":
            impression = elem.text if elem.text else ""

    # Extract all linked images
    for img in root.iter("parentImage"):

        image_name = img.attrib["id"] + ".png"
        image_path = os.path.join(IMAGE_PATH, image_name)

        records.append({
            "report_id": xml_file.replace(".xml", ""),
            "image_name": image_name,
            "image_path": image_path,
            "findings": findings,
            "impression": impression
        })

iuxray_metadata = pd.DataFrame(records)

iuxray_metadata.head()

In [ ]:
print(iuxray_metadata.shape)
iuxray_metadata.head()
iuxray_metadata.sample(5)
iuxray_metadata.info()
print(iuxray_metadata.isnull().sum())

In [ ]:
iuxray_metadata = iuxray_metadata[
    [
        "image_name",
        "image_path",
        "report_id",
        "findings",
        "impression"
    ]
]

In [ ]:
iuxray_metadata.sample(5)

In [ ]:
iuxray_metadata.to_csv("iuxray_metadata.csv", index=False)

print("IU-Xray metadata saved successfully!")

# COVIDx metadata creation

In [ ]:
#covidx metadata

In [ ]:
COVID_PATH = "/kaggle/input/datasets/andyczhao/covidx-cxr2"

train_file = os.path.join(COVID_PATH, "train.txt")
test_file = os.path.join(COVID_PATH, "test.txt")
val_file = os.path.join(COVID_PATH, "val.txt")

In [ ]:
with open(train_file, "r") as f:
    for _ in range(5):
        print(f.readline())

In [ ]:
with open(val_file, "r") as f:
    for _ in range(5):
        print(f.readline())

In [ ]:
with open(test_file, "r") as f:
    for _ in range(5):
        print(f.readline())

In [ ]:
import pandas as pd
import os
def load_split(txt_file, split_name):
    
    records = []

    with open(txt_file, "r") as f:

        for line in f:

            patient_id, image_name, label, source = line.strip().split()

            image_path = os.path.join(
                COVID_PATH,
                split_name,
                image_name
            )

            records.append({
                "patient_id": patient_id,
                "image_name": image_name,
                "image_path": image_path,
                "label": label,
                "source": source,
                "split": split_name
            })

    return pd.DataFrame(records)

In [ ]:
train_df = load_split(train_file, "train")

val_df = load_split(val_file, "val")

test_df = load_split(test_file, "test")

In [ ]:
covidx_metadata = pd.concat(
    [train_df, val_df, test_df],
    ignore_index=True
)

covidx_metadata.head()

In [ ]:
print(covidx_metadata.shape)

covidx_metadata.info()

print(covidx_metadata.isnull().sum())

covidx_metadata.sample(5)

In [ ]:
covidx_metadata.to_csv(
    "covidx_metadata.csv",
    index=False
)

print("COVIDx metadata saved successfully!")

# metadata cleaning

In [ ]:
nih_metadata = pd.read_csv("nih_metadata.csv")
iuxray_metadata = pd.read_csv("iuxray_metadata.csv")
covidx_metadata = pd.read_csv("covidx_metadata.csv")

In [ ]:
print("NIH Shape:", nih_metadata.shape)
print("IU-XRay Shape:", iuxray_metadata.shape)
print("COVIDx Shape:", covidx_metadata.shape)

In [ ]:
print("NIH")
print(nih_metadata.isnull().sum())

print("\nIU-XRay")
print(iuxray_metadata.isnull().sum())

print("\nCOVIDx")
print(covidx_metadata.isnull().sum())

In [ ]:
nih_metadata.drop_duplicates(inplace=True)

iuxray_metadata.drop_duplicates(inplace=True)

covidx_metadata.drop_duplicates(inplace=True)

In [ ]:
print("NIH:", nih_metadata.duplicated().sum())

print("IU-XRay:", iuxray_metadata.duplicated().sum())

print("COVIDx:", covidx_metadata.duplicated().sum())

In [ ]:
print(
    "Missing NIH Images:",
    (~nih_metadata["image_path"].apply(os.path.exists)).sum()
)

In [ ]:
print(
    "Missing IU Images:",
    (~iuxray_metadata["image_path"].apply(os.path.exists)).sum()
)

In [ ]:
print(
    "Missing COVIDx Images:",
    (~covidx_metadata["image_path"].apply(os.path.exists)).sum()
)

# Metadata Standardization

In [ ]:
nih_metadata = nih_metadata.rename(columns={
    "disease_labels": "label"
})

nih_metadata["dataset"] = "NIH"

nih_metadata.head()

In [ ]:
iuxray_metadata["dataset"] = "IU-XRay"

iuxray_metadata.head()

In [ ]:
covidx_metadata["dataset"] = "COVIDx"

covidx_metadata.head()

In [ ]:
print("NIH Columns:")
print(nih_metadata.columns)

print("\nIU-XRay Columns:")
print(iuxray_metadata.columns)

print("\nCOVIDx Columns:")
print(covidx_metadata.columns)

In [ ]:
nih_metadata.to_csv(
    "nih_metadata_standardized.csv",
    index=False
)

iuxray_metadata.to_csv(
    "iuxray_metadata_standardized.csv",
    index=False
)

covidx_metadata.to_csv(
    "covidx_metadata_standardized.csv",
    index=False
)

print("All standardized metadata saved successfully!")

# Image Preprocessing

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
def preprocess_image(image_path, img_size=(224, 224)):

    # read image in grayscale
    image = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)

    # resize
    image = cv2.resize(image, img_size)

    # clahe
    clahe = cv2.createCLAHE(
        clipLimit=2.0,
        tileGridSize=(8,8)
    )

    image = clahe.apply(image)

    # normalize
    image = image.astype(np.float32) / 255.0

    return image

In [ ]:
original = cv2.imread(sample_path, cv2.IMREAD_GRAYSCALE)

processed = preprocess_image(sample_path)

plt.figure(figsize=(10,5))

plt.subplot(1,2,1)
plt.imshow(original, cmap="gray")
plt.title("Original")
plt.axis("off")

plt.subplot(1,2,2)
plt.imshow(processed, cmap="gray")
plt.title("Processed")
plt.axis("off")

plt.show()

In [ ]:
sample_path = iuxray_metadata.loc[0, "image_path"]

original = cv2.imread(sample_path, cv2.IMREAD_GRAYSCALE)
processed = preprocess_image(sample_path)

plt.figure(figsize=(10,5))

plt.subplot(1,2,1)
plt.imshow(original, cmap="gray")
plt.title("Original IU-XRay")
plt.axis("off")

plt.subplot(1,2,2)
plt.imshow(processed, cmap="gray")
plt.title("Processed IU-XRay")
plt.axis("off")

plt.show()

In [ ]:
sample_path = covidx_metadata.loc[0, "image_path"]

original = cv2.imread(sample_path, cv2.IMREAD_GRAYSCALE)
processed = preprocess_image(sample_path)

plt.figure(figsize=(10,5))

plt.subplot(1,2,1)
plt.imshow(original, cmap="gray")
plt.title("Original COVIDx")
plt.axis("off")

plt.subplot(1,2,2)
plt.imshow(processed, cmap="gray")
plt.title("Processed COVIDx")
plt.axis("off")

plt.show()

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

In [ ]:
class ChestXrayDataset(Dataset):

    def __init__(self, dataframe, transform=None):
        self.dataframe = dataframe
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):

        row = self.dataframe.iloc[idx]

        image = preprocess_image(row["image_path"])

        if self.transform:
            image = self.transform(image)

        sample = {
            "image": image,
            "metadata": row.to_dict()
        }

        return sample

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor()
])

In [ ]:
nih_dataset = ChestXrayDataset(
    nih_metadata,
    transform=transform
)
print(len(nih_dataset))

In [ ]:
sample = nih_dataset[0]

print(sample.keys())

In [ ]:
print(sample["image"].shape)

In [ ]:
print(sample["metadata"])

# creating dataloader

In [ ]:
nih_dataset = ChestXrayDataset(
    nih_metadata,
    transform=transform
)

iuxray_dataset = ChestXrayDataset(
    iuxray_metadata,
    transform=transform
)

covidx_dataset = ChestXrayDataset(
    covidx_metadata,
    transform=transform
)

In [ ]:
nih_loader = DataLoader(
    nih_dataset,
    batch_size=16,
    shuffle=True,
    num_workers=2
)

iuxray_loader = DataLoader(
    iuxray_dataset,
    batch_size=16,
    shuffle=True,
    num_workers=2
)

covidx_loader = DataLoader(
    covidx_dataset,
    batch_size=16,
    shuffle=True,
    num_workers=2
)

In [ ]:
batch = next(iter(nih_loader))

print(batch["image"].shape)